<a href="https://colab.research.google.com/github/Shun0212/CodeBERTPretrained/blob/main/CodeMorph_ModernBERT_exp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -U transformers>=4.48.0
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.9 MB/s eta 0:00:00


In [2]:
!pip install transformers huggingface_hub
from huggingface_hub import notebook_login

# Hugging Face にログイン
notebook_login()


In [7]:
from transformers import AutoModel, AutoTokenizer
import os
import sys
import torch
import random
from datasets import load_dataset, Dataset
from tokenizers import BertWordPieceTokenizer
from transformers import ModernBertConfig, ModernBertForMaskedLM, PreTrainedTokenizerFast, DataCollatorForLanguageModeling, Trainer, TrainingArguments


# モデル名
repo_name = "Shuu12121/CodeMorph-ModernBERT"
# Hugging Face からモデルをロード
model = ModernBertForMaskedLM.from_pretrained(repo_name)
tokenizer = AutoTokenizer.from_pretrained(repo_name)

print("モデルのロード成功！")
print(model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/609M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/408k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, bias=False)
        )
      )
     

ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, bias=False)
        )
      )
      (1-11): 11

In [12]:
import torch
import numpy as np
import random
import re
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, T5ForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    if hasattr(model, "model"):
        outputs = model.model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "bert"):
        outputs = model.bert(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "roberta"):
        outputs = model.roberta(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "encoder"):
        # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
        outputs = model.encoder(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)
    else:
        outputs = model(**inputs)
        if hasattr(outputs, "last_hidden_state"):
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(outputs, "hidden_states"):
            embedding = outputs.hidden_states[-1][:, 0, :]
        else:
            raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")
    for code in all_codes:
        emb = get_cls_embedding(model, tokenizer, code, device)
        all_code_embeddings.append(emb)
    all_code_embeddings = np.concatenate(all_code_embeddings, axis=0)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)
    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデルでコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
    tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
    model_demo.to(device)
    print("\n【ModernBERT 単体ロード確認】")
    print("モデルのロード成功！")
    print(model_demo)

    # ------------------------------
    # ② 評価データセットのロード
    # ------------------------------
    print("\ngoogle/code_x_glue_ct_code_to_text データセット (Test) をロードします...")
    dataset = load_dataset("google/code_x_glue_ct_code_to_text", "python", split="test", trust_remote_code=True)
    max_examples = 100  # 実験用に先頭100サンプルを利用
    subset = dataset.select(range(max_examples))

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
        {"name": "microsoft/graphcodebert-base", "class": AutoModelForMaskedLM},
        {"name": "microsoft/codebert-base-mlm", "class": AutoModelForMaskedLM},
        {"name": "Salesforce/codet5p-220m-py", "class": T5ForConditionalGeneration},
        {"name": "Salesforce/codet5-large-ntp-py", "class": T5ForConditionalGeneration},
        {"name": "Shuu12121/CodeMorph-BERT", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-BERTv2", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT", "class": AutoModelForMaskedLM},
    ]

    for config in model_configs:
        model_name = config["name"]
        model_class = config["class"]
        print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = model_class.from_pretrained(model_name)
        model.to(device)

        metrics = evaluate_code_search(model, tokenizer, subset, device,
                                       max_examples=max_examples,
                                       pool_size=100,
                                       query_field="docstring",
                                       code_field="code")
        display_code_search_results(metrics, model_name)


使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, 

Some weights of the model checkpoint at microsoft/codebert-base-mlm were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/codebert-base-mlm Code Search Evaluation Results ====
MRR:         0.3789
MAP:         0.3789
R-Precision: 0.2900

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.2900    0.2900    0.2900    0.2900    0.2900         0.2900         
5     0.4500    0.3472    0.3726    0.3720    0.4500         0.4500         
10    0.5400    0.3582    0.4006    0.3915    0.5400         0.5400         
50    0.9400    0.3780    0.4905    0.4290    0.9400         0.9400         
100   1.0000    0.3789    0.5005    0.4309    1.0000         1.0000         

Salesforce/codet5p-220m-py を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Salesforce/codet5p-220m-py Code Search Evaluation Results ====
MRR:         0.5461
MAP:         0.5461
R-Precision: 0.4100

K     Re

In [ ]:
import torch
import numpy as np
import random
import re
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, T5ForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    if hasattr(model, "model"):
        outputs = model.model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "bert"):
        outputs = model.bert(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "roberta"):
        outputs = model.roberta(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "encoder"):
        # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
        outputs = model.encoder(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)
    else:
        outputs = model(**inputs)
        if hasattr(outputs, "last_hidden_state"):
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(outputs, "hidden_states"):
            embedding = outputs.hidden_states[-1][:, 0, :]
        else:
            raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")
    for code in all_codes:
        emb = get_cls_embedding(model, tokenizer, code, device)
        all_code_embeddings.append(emb)
    all_code_embeddings = np.concatenate(all_code_embeddings, axis=0)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)
    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデルでコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
    tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
    model_demo.to(device)
    print("\n【ModernBERT 単体ロード確認】")
    print("モデルのロード成功！")
    print(model_demo)

    # ------------------------------
    # ② 評価データセットのロード
    # ------------------------------
    print("\ngoogle/code_x_glue_ct_code_to_text データセット (Test) をロードします...")
    dataset = load_dataset("google/code_x_glue_ct_code_to_text", "python", split="test", trust_remote_code=True)
    max_examples = 14918  # 実験用に先頭100サンプルを利用
    subset = dataset.select(range(max_examples))

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
        {"name": "microsoft/graphcodebert-base", "class": AutoModelForMaskedLM},
        {"name": "microsoft/codebert-base-mlm", "class": AutoModelForMaskedLM},
        {"name": "Salesforce/codet5p-220m-py", "class": T5ForConditionalGeneration},
        {"name": "Salesforce/codet5-large-ntp-py", "class": T5ForConditionalGeneration},
        {"name": "Shuu12121/CodeMorph-BERT", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-BERTv2", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT", "class": AutoModelForMaskedLM},
    ]

    for config in model_configs:
        model_name = config["name"]
        model_class = config["class"]
        print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = model_class.from_pretrained(model_name)
        model.to(device)

        metrics = evaluate_code_search(model, tokenizer, subset, device,
                                       max_examples=max_examples,
                                       pool_size=100,
                                       query_field="docstring",
                                       code_field="code")
        display_code_search_results(metrics, model_name)


使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, 

Some weights of the model checkpoint at microsoft/codebert-base-mlm were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
